In [0]:

pii_df = spark.read.table("databricksbootcamp.greengrid.bronze_pii")
display(pii_df)

MeterID,PlanID,CustomerEmail
M-101,P2,kaitlynbrandon0@example.org
M-102,P2,omarsilva1@example.org
M-103,P1,kaitlynfrost2@example.org
M-104,P1,juliabauer3@example.org
M-105,P1,rossmclaughlin4@example.org
M-106,P1,omaredwards5@example.org
M-107,P1,ivankim6@example.org
M-108,P2,omarcosta7@example.org
M-109,P2,rosspatel8@example.org
M-110,P2,saracohen9@example.org


In [0]:
plans_df = spark.read.table("databricksbootcamp.greengrid.bronze_plans")
display(plans_df)
readings_df = spark.read.table("databricksbootcamp.greengrid.bronze_readings")
display(readings_df)

PlanID,PlanName,BasePricePerKwh
P1,Saver,0.1
P2,Eco-Green,0.25


kwh,meter_id,status,timestamp
9.33,M-148,OK,2025-12-17T12:16:55.000Z
7.49,M-163,OK,2025-12-08T18:09:01.000Z
4.6,M-153,OK,2025-12-31T13:39:24.000Z
0.68,M-109,OK,2025-12-23T19:23:29.000Z
11.25,M-101,OK,2025-12-27T01:15:03.000Z
3.8,M-127,OK,2025-12-18T22:27:03.000Z
3.77,M-110,OK,2025-12-26T01:50:53.000Z
11.84,M-195,OK,2025-12-09T23:10:57.000Z
0.28,M-154,OK,2025-12-27T12:55:23.000Z
5.26,M-172,OK,2025-12-13T17:46:40.000Z


In [0]:
from pyspark.sql.functions import *

In [0]:
display(
    readings_df.filter((col("kwh") > 0) & (col("kwh") < 100))
    .join(pii_df, readings_df.meter_id == pii_df.MeterID, "left")
    .withColumn("CustomerHash", sha2(col("CustomerEmail"), 256))
    .drop("CustomerEmail")
    
)



kwh,meter_id,status,timestamp,MeterID,PlanID,CustomerHash
9.33,M-148,OK,2025-12-17T12:16:55.000Z,M-148,P2,9cb3a91f484cc4891e98ada2597eed617e433a36c441a3cae18f724fee1cede8
7.49,M-163,OK,2025-12-08T18:09:01.000Z,M-163,P1,61d7dc9ecbfd418770f1bb8d0af83b73b6c5799afac53e814f7e3b6de438c4ff
4.6,M-153,OK,2025-12-31T13:39:24.000Z,M-153,P1,190000b2c9809fd826208961d5b5e212999af2f1ee7941195cb8cf74daf2306d
0.68,M-109,OK,2025-12-23T19:23:29.000Z,M-109,P2,2f88faca323e616212cc1615923e63a8d59054824389e7376528dc5ccb8ccc71
11.25,M-101,OK,2025-12-27T01:15:03.000Z,M-101,P2,19c5f9b1b552ebb9adbb7e62a91915ca7122f402aa7dbbee8782a42743a48350
3.8,M-127,OK,2025-12-18T22:27:03.000Z,M-127,P1,0a4648e8f79c9615038a116fd6f220f87ce544fbdf4f913d7dff3fe6aa101e5f
3.77,M-110,OK,2025-12-26T01:50:53.000Z,M-110,P2,c6f946ae5019d06249191ebf7dd2d322d30dccb10d8a8abf7227a82f4429006b
11.84,M-195,OK,2025-12-09T23:10:57.000Z,M-195,P2,dc3d473c6dc96bafb81395df53bfd8ffce115f2b70f841803a68a546a8dfbbe5
0.28,M-154,OK,2025-12-27T12:55:23.000Z,M-154,P1,e20f6101d9700462b9301909f6db50b86c1e7ee7380d388df6b0b13b8ee0808c
5.26,M-172,OK,2025-12-13T17:46:40.000Z,M-172,P2,065f3a9dbf9f0fd1dc0923609df97ac6f5b10aaa8c460c0f5cf1fbb25fd1b285


In [0]:
dbutils.widgets.text("peak_multiplier", "2","1. Peak Multipier")
peak_multiplier = dbutils.widgets.get("peak_multiplier")
#peak_multiplier = 2

In [0]:
dbutils.widgets.dropdown("2. From","18",["15","16","17","18","19","20"])
from_peak = dbutils.widgets.get("From")

dbutils.widgets.dropdown("3. To","21",["17","18","19","20","21","22"])
to_peak = dbutils.widgets.get("To")

In [0]:
print(from_peak)
print(to_peak)

18
21


In [0]:
silver_df = (
    readings_df.filter((col("kwh") > 0) & (col("kwh") < 100))
    .join(pii_df, readings_df.meter_id == pii_df.MeterID, "left")
    .withColumn("CustomerHash", sha2(col("CustomerEmail"), 256))
    .drop("CustomerEmail")
    .join(plans_df,"PlanID","left")
    .drop("CustomerEmail","MeterID")
    .withColumn("Hour",hour(col("timestamp")))
    .withColumn("FinalRate",when((col("Hour")>=18) & (col("Hour")<=21),col("BasePricePerKwh")*peak_multiplier)
                .otherwise(col("BasePricePerKwh")))
    .withColumn("BillAmount",round(col("kwh")*col("FinalRate"),2))
)

    

In [0]:
silver_df.write.mode("overwrite").saveAsTable("databricksbootcamp.greengrid.silver_smartbill")

In [0]:
readings_df = spark.read.table("databricksbootcamp.greengrid.bronze_readings")

quarantine_df = readings_df.filter((col("kwh") <= 0) | (col("kwh") > 100)).withColumn("Rejection_Reason",
    when(col("kwh") < 0, "Negative_readings").otherwise("Overload"))

display(quarantine_df)


kwh,meter_id,status,timestamp,Rejection_Reason
1232.14,M-187,OK,2025-12-28T22:55:00.000Z,Overload
464.48,M-163,OK,2025-12-08T00:29:00.000Z,Overload
-2.32,M-114,ERR,2025-12-19T04:47:00.000Z,Negative_readings
2347.24,M-115,OK,2025-12-31T20:00:00.000Z,Overload
-5.88,M-190,ERR,2025-12-09T17:53:00.000Z,Negative_readings
1056.24,M-138,OK,2025-12-25T20:35:00.000Z,Overload
1157.25,M-121,OK,2025-12-07T22:17:00.000Z,Overload
-7.23,M-114,ERR,2025-12-04T05:10:00.000Z,Negative_readings
-5.52,M-115,ERR,2025-12-05T04:31:00.000Z,Negative_readings
4779.87,M-191,OK,2025-12-13T06:04:00.000Z,Overload


In [0]:
quarantine_df.write.mode("overwrite").saveAsTable("databricksbootcamp.greengrid.silver_rejected")